In [1]:
import pandas as pd

In [2]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


In [ ]:
from models import Ride, ride_from_row, ride_serializer

In [ ]:
# from dataclasses import dataclass

# # create a class to ensure datatypes utilized
# @dataclass
# class Ride:
#     PULocationID: int
#     DOLocationID: int
#     trip_distance: float
#     total_amount: float
#     tpep_pickup_datetime: int  # epoch milliseconds

# now being imported from models.py

In [ ]:
# # function to convert a df row into a "ride"
# # transforming timestamp into epoch milliseconds for Flink later
# def ride_from_row(row):
#     return Ride(
#         PULocationID=int(row['PULocationID']),
#         DOLocationID=int(row['DOLocationID']),
#         trip_distance=float(row['trip_distance']),
#         total_amount=float(row['total_amount']),
#         tpep_pickup_datetime=int(row['tpep_pickup_datetime'].timestamp() * 1000),
#     )

# now being imported from models.py

In [6]:
ride = ride_from_row(df.iloc[0])
ride

# note: our data is currently a python object
# need to transform it in order to send it to kafka

Ride(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)

In [ ]:
from kafka import KafkaProducer
import json
import dataclasses

In [ ]:
# serializer to send one line of code - used to test
def json_serializer(data):
    return json.dumps(data).encode('utf-8')

# server = 'localhost:9092'    # did not work - had to insert text manually below
# necessary to specify when running outside Docker - can use multiple (production)
# included in docker-compose.yaml

# create a kafka producer
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
      # where "broker" accepts connections
    value_serializer=json_serializer
      # serializer to convert python to json - kafka uses raw bytes
)

topic_name = 'rides'
# send data - as a row of bytes (in json format)
producer.send(topic_name, value=dataclasses.asdict(ride))
producer.flush()

In [ ]:
# # function to incorporate dataclasses directly
# def ride_serializer(ride):
#     ride_dict = dataclasses.asdict(ride)
#     json_str = json.dumps(ride_dict)
#     return json_str.encode('utf-8')
# now being imported from models.py

# updated producer serializer
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=ride_serializer
)

# send updated script
producer.send(topic_name, value=ride)
producer.flush()